# MCQA — task demo

Three sections:

1. **Causal model** — variables, mechanisms, sampled prompt.
2. **Templates & token positions** — how the choice symbols are located in the tokenized prompt, including the dynamic `correct_symbol` position.
3. **Counterfactual generators** — what changes between input and counterfactual under each generator.

Tokenization uses `gpt2` to keep the demo cheap and offline. No interventions are run — for that, see `analyses/locate/demo.ipynb`.

In [1]:
from causalab.tasks.MCQA.causal_models import (
    positional_causal_model,
    NUM_CHOICES,
    OBJECTS,
    COLORS,
)
from causalab.tasks.MCQA.counterfactuals import (
    sample_answerable_question,
    different_symbol,
    same_symbol_different_position,
    random_counterfactual,
)
from causalab.tasks.MCQA.token_positions import create_token_positions

model = positional_causal_model
f"NUM_CHOICES={NUM_CHOICES}, |OBJECTS|={len(OBJECTS)}, |COLORS|={len(COLORS)}"

'NUM_CHOICES=2, |OBJECTS|=10, |COLORS|=10'

## 1. Causal model

Inputs: `template`, `object`, `color`, plus `choice0`/`choice1` (color labels) and `symbol0`/`symbol1` (letters). Computed: `raw_input`, `answer_position`, `answer`, `raw_output`. Sample one answerable question and inspect the trace.

In [2]:
import random

random.seed(0)
trace = sample_answerable_question()  # already fully traced

print("Inputs:")
print(f"  object  = {trace['object']!r}")
print(f"  color   = {trace['color']!r}")
for i in range(NUM_CHOICES):
    print(
        f"  choice{i} = {trace[f'choice{i}']!r:<10}  symbol{i} = {trace[f'symbol{i}']!r}"
    )
print()
print("Computed:")
print(f"  answer_position = {trace['answer_position']}")
print(f"  answer          = {trace['answer']!r}")
print(f"  raw_output      = {trace['raw_output']!r}")
print()
print("raw_input:")
print(trace["raw_input"])

Inputs:
  object  = 'cup'
  color   = 'red'
  choice0 = 'orange'    symbol0 = 'M'
  choice1 = 'red'       symbol1 = 'Z'

Computed:
  answer_position = 1
  answer          = 'Z'
  raw_output      = ' Z'

raw_input:
The cup is red. What color is the cup?
M. orange
Z. red
Answer:


## 2. Templates & token positions

`create_token_positions(pipeline)` returns positions for each choice's symbol, the period after each symbol, the dynamic `correct_symbol` (and its period), and `last_token`. The dynamic positions read `color` / `choice{i}` from the input sample to pick the correct choice, so we don't need to know the answer ahead of time.

In [3]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1)
positions = create_token_positions(pipeline)
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['correct_symbol',
 'correct_symbol_period',
 'symbol0',
 'symbol0_period',
 'symbol1',
 'symbol1_period',
 'last_token']

In [4]:
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id

print(f"correct symbol = {trace['answer']!r} at position {trace['answer_position']}")
print()
print(f"{'idx':>4}  {'token':<20}  positions")
for i, tok in enumerate(decoded):
    if ids[i] == pad_id:
        continue
    hits = [name for name, pos in positions.items() if i in pos.index(trace)]
    marker = ", ".join(hits) if hits else ""
    print(f"{i:>4}  {tok!r:<20}  {marker}")

correct symbol = 'Z' at position 1

 idx  token                 positions
   0  'The'                 
   1  ' cup'                
   2  ' is'                 
   3  ' red'                
   4  '.'                   
   5  ' What'               
   6  ' color'              
   7  ' is'                 
   8  ' the'                
   9  ' cup'                
  10  '?'                   
  11  '\n'                  
  12  'M'                   symbol0
  13  '.'                   symbol0_period
  14  ' orange'             
  15  '\n'                  
  16  'Z'                   correct_symbol, symbol1
  17  '.'                   correct_symbol_period, symbol1_period
  18  ' red'                
  19  '\n'                  
  20  'Answer'              
  21  ':'                   last_token


Note that `correct_symbol` lands on the same index as one of `symbol0` / `symbol1` — whichever corresponds to the matching color. Re-sample to confirm it tracks the answer:

In [5]:
for _ in range(3):
    t = sample_answerable_question()
    correct_idx = positions["correct_symbol"].index(t)[0]
    sym0_idx = positions["symbol0"].index(t)[0]
    sym1_idx = positions["symbol1"].index(t)[0]
    print(
        f"answer_position={t['answer_position']}  answer={t['answer']!r}  "
        f"correct_symbol@{correct_idx}  symbol0@{sym0_idx}  symbol1@{sym1_idx}"
    )

answer_position=0  answer='J'  correct_symbol@12  symbol0@12  symbol1@16
answer_position=0  answer='J'  correct_symbol@12  symbol0@12  symbol1@16
answer_position=0  answer='L'  correct_symbol@12  symbol0@12  symbol1@16


## 3. Counterfactual generators

MCQA ships with three generators. The default for `generate_dataset` is `different_symbol` because it deconfounds `answer_position` (the typical target variable) from `answer` (the surface token). Each call below produces one `{input, counterfactual_inputs}` pair; we render both side-by-side.

In [6]:
def show_pair(label, ex):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- {label} ---")
    print(
        f"  base : color={base['color']:<8} symbols=({base['symbol0']}, {base['symbol1']}) "
        f"choices=({base['choice0']}, {base['choice1']})  -> answer_position={base['answer_position']}, answer={base['answer']!r}"
    )
    print(
        f"  cf   : color={cf['color']:<8} symbols=({cf['symbol0']}, {cf['symbol1']}) "
        f"choices=({cf['choice0']}, {cf['choice1']})  -> answer_position={cf['answer_position']}, answer={cf['answer']!r}"
    )
    print()


random.seed(1)
show_pair("different_symbol", different_symbol())
show_pair("same_symbol_different_position", same_symbol_different_position())
show_pair("random_counterfactual", random_counterfactual())

--- different_symbol ---
  base : color=blue     symbols=(P, Y) choices=(blue, orange)  -> answer_position=0, answer='P'
  cf   : color=blue     symbols=(O, Q) choices=(blue, orange)  -> answer_position=0, answer='O'

--- same_symbol_different_position ---
  base : color=blue     symbols=(M, N) choices=(blue, black)  -> answer_position=0, answer='M'
  cf   : color=blue     symbols=(N, M) choices=(black, blue)  -> answer_position=1, answer='M'

--- random_counterfactual ---
  base : color=yellow   symbols=(A, U) choices=(blue, yellow)  -> answer_position=1, answer='U'
  cf   : color=yellow   symbols=(Q, H) choices=(yellow, brown)  -> answer_position=0, answer='Q'



Reading the output:

- **`different_symbol`** — `answer_position` is identical, `answer` letter changes. Use this when the analysis targets `answer_position`.
- **`same_symbol_different_position`** — symbol set is identical, `answer_position` flips. Use this when targeting `answer` (token identity).
- **`random_counterfactual`** — every input variable may differ. Useful for distribution-level baselines.

## Next steps

Run `locate` end-to-end via Hydra:

```bash
./scripts/run_exp.sh mcqa_locate
```

Outputs land under `artifacts/MCQA/<model>/locate/...`.